# PACE Inference Server — Playbook

Interactive walkthrough of the AMD PACE inference server covering:

1. **Health & Readiness** — verify the server, scheduler, models, and queue
2. **Non-Streaming Completion** — single-shot request/response
3. **Streaming Completion (SSE)** — token-by-token output via Server-Sent Events
4. **MLPerf Mode** — raw token-ID output for throughput benchmarking
5. **Teardown** — how to shut down and clean up

---

### Prerequisites

**Kernel setup** — register the PACE conda environment as a Jupyter kernel (one-time):

```bash
conda activate pace-env-py3.12
pip install ipykernel
python -m ipykernel install --user --name pace-env-py3.12 --display-name "PACE (py3.12)"
```

Then select **PACE (py3.12)** from the kernel picker (top-right corner in Jupyter).

**Start the server** by running the cell below, or in a separate terminal:

```bash
conda activate pace-env-py3.12
pace-server --server_model meta-llama/Llama-3.1-8B-Instruct \
  --dtype bfloat16 --scheduler_metrics_enabled True --enable_prometheus
```


## Setup

Imports and configuration. Change `MODEL` or `ROUTER_URL` here if your setup differs.

In [ ]:
# ******************************************************************************
# Copyright (c) 2026 Advanced Micro Devices, Inc.
# All rights reserved.
# Portion of this file may consist of AI-generated code.
# ******************************************************************************

import json
import os
import sys
import time

import httpx

# Ensure the conda env's binaries (pace-server, etc.) are on PATH
CONDA_PREFIX = os.environ.get("CONDA_PREFIX", "")
if CONDA_PREFIX:
    conda_bin = os.path.join(CONDA_PREFIX, "bin")
    if conda_bin not in os.environ["PATH"]:
        os.environ["PATH"] = conda_bin + ":" + os.environ["PATH"]

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
ROUTER_URL = "http://localhost:8080"

LAUNCH_CMD = (
    f"pace-server --server_model {MODEL} --dtype bfloat16 \\\n"
    f"  --scheduler_metrics_enabled True --enable_prometheus"
)

print(f"Model : {MODEL}")
print(f"Router: {ROUTER_URL}")
print(
    f"PATH includes: {CONDA_PREFIX}/bin"
    if CONDA_PREFIX
    else "Warning: CONDA_PREFIX not set"
)

## Start Server (optional)

Run this cell to launch the server **in the background** from within the notebook via `subprocess.Popen`. The process runs independently and logs to `pace_server.log`.

Skip this cell if you already have a server running in a separate terminal.

In [ ]:
import subprocess

CONDA_ENV = "pace-env-py3.12"

server_cmd = (
    f"conda run --no-capture-output -n {CONDA_ENV} "
    f"pace-server --server_model {MODEL} --dtype bfloat16 "
    f"--scheduler_metrics_enabled True --enable_prometheus"
)

server_proc = subprocess.Popen(
    server_cmd,
    shell=True,
    stdout=open("pace_server.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"Server starting (PID {server_proc.pid})...")
print(f"Command: {server_cmd}")
print(f"Logs:    pace_server.log")
print("\nWait ~30-60s for model loading, then run the Server Check cell below.")

## Server Check

Verify the server is reachable, the scheduler is running, and the expected model is loaded.

A lightweight probe request (`max_new_tokens=1`) confirms the model. If the server isn't running or the wrong model is loaded, the cell prints the exact launch command you need.

In [ ]:
async def check_server():
    try:
        async with httpx.AsyncClient(timeout=10.0) as client:
            resp = await client.get(f"{ROUTER_URL}/v1/health")
            health = resp.json()
    except Exception as e:
        print(f"Cannot reach server at {ROUTER_URL}: {e}")
        print(f"\nStart the server with:\n\n    {LAUNCH_CMD}\n")
        return False

    if not health.get("scheduler_running"):
        print(f"Scheduler not running.\n\n    {LAUNCH_CMD}\n")
        return False

    try:
        async with httpx.AsyncClient(timeout=60.0) as client:
            probe = await client.post(
                f"{ROUTER_URL}/v1/completions",
                json={
                    "model": MODEL,
                    "prompt": "probe",
                    "stream": False,
                    "max_tokens": 1,
                },
            )
            if (
                probe.status_code == 404
                and probe.json().get("error", {}).get("code") == "model_not_found"
            ):
                print(f"Model '{MODEL}' not loaded on server.")
                print(f"\nStart the server with:\n\n    {LAUNCH_CMD}\n")
                return False
    except Exception:
        pass

    print(f"Server healthy (queue={health.get('queue_size', 0)})")
    return True


assert await check_server(), "Server not ready — see message above."

## 1. Health & Readiness Checks

Three endpoints give you full visibility into server state:

| Endpoint | Purpose |
|---|---|
| `GET /v1/health` | Scheduler status, queue depth, active requests |
| `GET /v1/models` | All models the server was configured to support |
| `GET /v1/queue/status` | Current queue size and active request count |

Run this cell to inspect all three.

In [ ]:
async with httpx.AsyncClient(timeout=30.0) as client:
    # Health
    print("GET /v1/health")
    health = (await client.get(f"{ROUTER_URL}/v1/health")).json()
    for k, v in health.items():
        print(f"  {k}: {v}")

    # Models
    print("\nGET /v1/models")
    models = (await client.get(f"{ROUTER_URL}/v1/models")).json()
    for m in models.get("data", []):
        print(f"  {m.get('id', '?')}  dtypes={m.get('dtypes', [])}")

    # Queue
    print("\nGET /v1/queue/status")
    qs = (await client.get(f"{ROUTER_URL}/v1/queue/status")).json()
    print(
        f"  queue_size={qs.get('queue_size', '?')}  active={qs.get('active_requests', '?')}"
    )

## 2. Non-Streaming Completion

A standard synchronous request: send a prompt, receive the full generated text in one response.

**Request format:**
```json
{
  "model": "meta-llama/Llama-3.1-8B-Instruct",
  "prompt": "...",
  "stream": false,
  "max_tokens": 50,
  "temperature": 0
}
```

**Response** contains `choices[0].text` with the generated text.

In [ ]:
payload = {
    "model": MODEL,
    "prompt": "What is the capital of France? Answer in one sentence.",
    "stream": False,
    "max_tokens": 50,
    "temperature": 0,
}

print(f"Prompt: {payload['prompt']}\n")

t0 = time.time()
async with httpx.AsyncClient(timeout=300.0) as client:
    resp = await client.post(f"{ROUTER_URL}/v1/completions", json=payload)
elapsed = time.time() - t0
result = resp.json()

if "choices" in result and result["choices"]:
    text = result["choices"][0].get("text", "")
    print(f"Output: {text}")
    print(f"\nTime: {elapsed:.2f}s  |  ID: {result.get('id', 'N/A')}")
else:
    print(f"Unexpected response: {json.dumps(result)}")

## 3. Streaming Completion (SSE)

With `"stream": true`, the server sends tokens as **Server-Sent Events** (`text/event-stream`).

Each SSE line looks like:
```
data: {"id":"cmpl-...","object":"text_completion","choices":[{"index":0,"text":"token_text"}]}
```

The stream ends with `data: [DONE]`.

This cell prints tokens live as they arrive and reports TTFT (time-to-first-token) and throughput.

In [ ]:
payload = {
    "model": MODEL,
    "prompt": "Write a short poem about parallel computing.",
    "stream": True,
    "max_tokens": 100,
    "temperature": 0,
}
hdrs = {"Content-Type": "application/json", "Accept": "text/event-stream"}

print(f"Prompt: {payload['prompt']}\n")
print("Output: ", end="", flush=True)

token_count = 0
t0 = time.time()
first_token = None

async with httpx.AsyncClient(timeout=300.0) as client:
    async with client.stream(
        "POST", f"{ROUTER_URL}/v1/completions", json=payload, headers=hdrs
    ) as resp:
        buf = ""
        async for chunk in resp.aiter_text():
            buf += chunk
            while "\n" in buf:
                line, buf = buf.split("\n", 1)
                line = line.strip()
                if not line or not line.startswith("data: "):
                    continue
                if line == "data: [DONE]":
                    break
                try:
                    obj = json.loads(line[6:])
                    content = obj.get("choices", [{}])[0].get("text", "")
                    if content:
                        if first_token is None:
                            first_token = time.time()
                        token_count += 1
                        print(content, end="", flush=True)
                except json.JSONDecodeError:
                    continue

elapsed = time.time() - t0
ttft = first_token - t0 if first_token else 0
print(
    f"\n\nTokens: {token_count}  |  TTFT: {ttft:.3f}s  |  "
    f"Total: {elapsed:.2f}s  |  {token_count / elapsed:.1f} tok/s"
)

## 4. MLPerf Mode — Token IDs Only

Setting `"mlperf_mode": true` skips text decoding entirely and returns **token IDs** as space-separated integers in `choices[0].text`. This is the fastest path and is used for throughput benchmarking.

### 4a. Non-Streaming MLPerf

Response contains `choices[0].text` with space-separated integer token IDs.

In [ ]:
payload = {
    "model": MODEL,
    "prompt": "The quick brown fox jumps over the lazy dog.",
    "stream": False,
    "mlperf_mode": True,
    "max_tokens": 20,
    "temperature": 0,
}

print(f"Prompt: {payload['prompt']}\n")

async with httpx.AsyncClient(timeout=300.0) as client:
    t0 = time.time()
    resp = await client.post(f"{ROUTER_URL}/v1/completions", json=payload)
    elapsed = time.time() - t0
    result = resp.json()

if "choices" in result and result["choices"]:
    text = result["choices"][0].get("text", "")
    ids = [int(x) for x in text.split() if x]
    print(f"Token IDs: {ids}")
    print(f"Count:     {len(ids)} tokens")
    print(f"Time:      {elapsed:.2f}s")
else:
    print(f"Unexpected response: {json.dumps(result)}")

### 4b. Streaming MLPerf

In streaming MLPerf mode, the server sends **plain text** — one integer token ID per line — instead of SSE JSON. The stream ends with `[DONE]`.

In [ ]:
payload["stream"] = True

async with httpx.AsyncClient(timeout=300.0) as client:
    t0 = time.time()
    first_token = None
    received = []
    async with client.stream(
        "POST", f"{ROUTER_URL}/v1/completions", json=payload
    ) as resp:
        buf = ""
        async for chunk in resp.aiter_text():
            buf += chunk
            while "\n" in buf:
                line, buf = buf.split("\n", 1)
                line = line.strip()
                if not line or line == "[DONE]":
                    continue
                try:
                    tid = int(line)
                    if first_token is None:
                        first_token = time.time()
                    received.append(tid)
                    print(f"  Token {len(received):>3}:  {tid}")
                except ValueError:
                    pass

    elapsed = time.time() - t0
    ttft = first_token - t0 if first_token else 0

print(f"\nAll IDs: {received}")
print(
    f"Count:   {len(received)} tokens  |  TTFT: {ttft:.3f}s  |  Total: {elapsed:.2f}s"
)

## 5. Teardown & Cleanup

Run the cell below to stop the server that was launched from this notebook. If you started the server in a separate terminal, press **Ctrl+C** there instead.

The server is **stateless** — restarting is safe at any time.

In [ ]:
import os, signal, subprocess as _sp

if "server_proc" in dir() and server_proc.poll() is None:
    pid = server_proc.pid
    print(f"Stopping server (PID {pid})...")
    try:
        pgid = os.getpgid(pid)
        os.killpg(pgid, signal.SIGTERM)
    except (ProcessLookupError, PermissionError):
        server_proc.terminate()
    try:
        server_proc.wait(timeout=20)
        print("Server stopped.")
    except _sp.TimeoutExpired:
        print("Graceful stop timed out, force killing...")
        try:
            os.killpg(os.getpgid(pid), signal.SIGKILL)
        except (ProcessLookupError, PermissionError):
            server_proc.kill()
        server_proc.wait(timeout=5)
        print("Server killed.")
else:
    print("No server process found (was it started from a terminal instead?)")

print("\nOptional cleanup:  rm -rf ~/.cache/pace/prometheus/")